### Import Libraries

In [5]:
import os
import cv2
import torch
import shutil
import random
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

np.random.seed(42)

### Mount google drive


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Oject Detection

## Avenue Dataset

In [6]:
import requests
import zipfile

# --- 1. DEFINE PATHS ON YOUR GOOGLE DRIVE ---
# We'll save the dataset in MyDrive/Colab_Data/Avenue_Dataset
base_drive_path = '/content/drive/MyDrive/Avenue'
# extract_path = os.path.join(base_drive_path, 'Avenue_Dataset')

# Create the base directory if it doesn't exist
os.makedirs(base_drive_path, exist_ok=True)

In [7]:
# --- 2. CHECK IF DATASET ALREADY EXISTS ---
if os.path.exists(base_drive_path) and os.listdir(base_drive_path):
    print(f"✅ Dataset already exists at: {base_drive_path}")
else:
    print(f"Dataset not found. Starting download to {base_drive_path}...")

    # --- 3. DOWNLOAD AND UNZIP ---
    url = "http://www.cse.cuhk.edu.hk/leojia/projects/detectabnormal/Avenue_Dataset.zip"
    zip_file_path = os.path.join(base_drive_path, "Avenue_Dataset.zip")

    try:
        # Download the file
        response = requests.get(url, stream=True)
        response.raise_for_status()
        with open(zip_file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Download complete.")

        # Unzip the file
        print("Unzipping...")
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(base_drive_path)
        print(f"✅ Successfully unzipped to: {base_drive_path}")

        # Clean up the zip file
        os.remove(zip_file_path)
        print("Removed temporary zip file.")

    except requests.exceptions.RequestException as e:
        print(f"❌ Error downloading file: {e}")

✅ Dataset already exists at: /content/drive/MyDrive/Avenue


## Git Clone

In [8]:
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5.git

In [9]:
%cd yolov5/

/content/yolov5


In [10]:
!pip install -r requirements.txt

In [11]:
%cd /content

/content


## Load Model and Detect

In [ ]:
%cd /content/yolov5

In [ ]:
model_path = "/content/drive/MyDrive/mot17_yolo/results_train/results_2/yolov5_mot17/weights/best.pt"
dataset_path = '/content/drive/MyDrive/Avenue/Avenue Dataset/testing_videos/01.avi'
output_path = '/content/drive/MyDrive/Avenue/Detection'

In [ ]:
!python detect.py --weights "{model_path}" \
                  --source "{dataset_path}" \
                  --project "{output_path}" \
                  --name "exp" \
                  --conf-thres 0.4 \
                  --iou-thres 0.45 \
                  --classes 0 \
                  --exist-ok \
                  --line-thickness 2 \
                  --hide-labels \
                  --hide-conf \
                  --save-txt

print(f"\n✅ Detection complete! Results are saved in: {output_path}/exp")

# Object Tracking

In [32]:
!pip install deep-sort-realtime opencv-python tqdm

In [33]:
from deep_sort_realtime.deepsort_tracker import DeepSort
from google.colab.patches import cv2_imshow

In [34]:
model_path = '/content/drive/MyDrive/mot17_yolo/results_train/results_2/yolov5_mot17/weights/best.pt'
video_path = '/content/drive/MyDrive/Avenue/Avenue Dataset/testing_videos/01.avi'
output_folder = '/content/drive/MyDrive/Avenue/Tracking'

output_video_path = os.path.join(output_folder, 'tracked_video.mp4')
txt_output_path = os.path.join(output_folder, 'tracking_results.txt') # Path for the text file

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

In [35]:
model = torch.hub.load('ultralytics/yolov5', 'custom', path=model_path, force_reload=True)
model.conf = 0.5
model.classes = [0]

tracker = DeepSort(max_age=10,
                   n_init=2,
                   max_iou_distance=0.5)

# 4. Process the Video
cap = cv2.VideoCapture(video_path)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

print("Processing video and performing tracking...")
frame_count = 0

### NEW ###
# Open the text file for writing the results
with open(txt_output_path, 'w') as f:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame)

        detections = []
        for det in results.xyxy[0]:
            x1, y1, x2, y2, conf, cls = det
            if int(cls) == 0:
                w, h = x2 - x1, y2 - y1
                bbox = [int(x1), int(y1), int(w), int(h)]
                detections.append((bbox, float(conf), 'pedestrian'))

        tracks = tracker.update_tracks(detections, frame=frame)

        for track in tracks:
            if not track.is_confirmed():
                continue

            track_id = track.track_id
            ltrb = track.to_ltrb()
            x1, y1, x2, y2 = map(int, ltrb)

            ### NEW ###
            # Write tracking data to the text file
            # Format: frame_id, track_id, left, top, width, height
            width = x2 - x1
            height = y2 - y1
            f.write(f'{frame_count},{track_id},{x1},{y1},{width},{height}\n')

            # Draw bounding box and ID on the frame
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"ID: {track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        writer.write(frame)
        frame_count += 1

# 5. Release Resources
cap.release()
writer.release()

print(f"✅ Tracking complete! Processed {frame_count} frames.")
print(f"Output video saved to: {output_video_path}")
print(f"Tracking results saved to: {txt_output_path}")

Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip
YOLOv5 🚀 2025-7-19 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


Processing video and performing tracking...
✅ Tracking complete! Processed 1439 frames.
Output video saved to: /content/drive/MyDrive/Avenue/Tracking/tracked_video.mp4
Tracking results saved to: /content/drive/MyDrive/Avenue/Tracking/tracking_results.txt


# Anomaly Detection






In [36]:
# --- Configuration ---
# Path to your tracking results text file
txt_path = '/content/drive/MyDrive/Avenue/Tracking/tracking_results.txt'
output_folder = '/content/drive/MyDrive/Avenue/Anomaly_Detection'
histogram_path = os.path.join(output_folder, 'speed_histogram.png')

# --- Step 1: Load and Process Tracking Data ---
print("Loading tracking data...")
df = pd.read_csv(txt_path, header=None, names=['frame', 'id', 'x', 'y', 'w', 'h'])

# Calculate the center point
df['center_x'] = df['x'] + df['w'] / 2
df['center_y'] = df['y'] + df['h'] / 2

# --- Step 2: Calculate Speed for Each Track ID ---
print("Calculating speeds...")
df['dx'] = df.groupby('id')['center_x'].diff()
df['dy'] = df.groupby('id')['center_y'].diff()
df['speed'] = np.hypot(df['dx'], df['dy'])

# --- Step 3: Plot the Histogram ---
print("Generating speed histogram...")
# We must drop NaN values before plotting. NaN occurs for the first frame of each track.
speeds = df['speed'].dropna()

plt.figure(figsize=(12, 6))
plt.hist(speeds, bins=200, alpha=0.75, color='royalblue')
plt.title('Distribution of Pedestrian Speeds', fontsize=16)
plt.xlabel('Speed (pixels/frame)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', alpha=0.5)
plt.minorticks_on()

# Save the plot to your Drive
plt.savefig(histogram_path, dpi=300)

# Show the plot in Colab
plt.show()

Loading tracking data...
Calculating speeds...
Generating speed histogram...


In [38]:
# --- Configuration ---
video_path = '/content/drive/MyDrive/Avenue/Avenue Dataset/testing_videos/01.avi'
txt_path = '/content/drive/MyDrive/Avenue/Tracking/tracking_results.txt'
output_folder = '/content/drive/MyDrive/Avenue/Anomaly_Detection'
output_video_path = os.path.join(output_folder, 'anomaly_video_final.mp4')

# --- Parameters ---
SPEED_THRESHOLD = 5

# --- Step 1: Load and Process Tracking Data ---
print("Loading tracking data...")
df = pd.read_csv(txt_path, header=None, names=['frame', 'id', 'x', 'y', 'w', 'h'])
df['center_x'] = df['x'] + df['w'] / 2
df['center_y'] = df['y'] + df['h'] / 2
df['dx'] = df.groupby('id')['center_x'].diff()
df['dy'] = df.groupby('id')['center_y'].diff()
df['speed'] = np.hypot(df['dx'], df['dy'])

# --- Step 2: Identify Anomalies based on AVERAGE Speed ---
print("Calculating average speeds and identifying anomalies...")
average_speeds = df.groupby('id')['speed'].mean()
anomalous_ids = set(average_speeds[average_speeds > SPEED_THRESHOLD].index)
total_ids = set(df['id'])

if len(total_ids) > 0:
    anomaly_percentage = (len(anomalous_ids) / len(total_ids)) * 100
    print(f"\nFound {len(anomalous_ids)} anomalous tracks based on average speed.")
    print(f"Percentage of IDs marked as anomalous: {anomaly_percentage:.2f}%")
else:
    print("No tracks found in the data.")


# --- Step 3: Visualize Results ---
print("Generating final anomaly video...")
cap = cv2.VideoCapture(video_path)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
frame_data = {frame: group for frame, group in df.groupby('frame')}

current_frame = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if current_frame in frame_data:
        tracks_in_frame = frame_data[current_frame]
        for _, track in tracks_in_frame.iterrows():
            track_id = int(track['id'])

            # ### UPDATED DRAWING LOGIC ###
            # Only draw anything if the track ID is in our anomalous set
            if track_id in anomalous_ids:
                x, y, w, h = int(track['x']), int(track['y']), int(track['w']), int(track['h'])
                color = (0, 0, 255)  # Red for anomaly

                # Draw the bounding box
                cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

                # Draw the text "Anomaly"
                cv2.putText(frame, "Anomaly", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    writer.write(frame)
    current_frame += 1

cap.release()
writer.release()
print(f"✅ Final video generation complete. Saved to: {output_video_path}")

Loading tracking data...
Calculating average speeds and identifying anomalies...

Found 8 anomalous tracks based on average speed.
Percentage of IDs marked as anomalous: 4.71%
Generating final anomaly video...
✅ Final video generation complete. Saved to: /content/drive/MyDrive/Avenue/Anomaly_Detection/anomaly_video_final.mp4
